# NLP Pipeline: Job Market Analysis
**Author:** Aby Joe Jose  
**Course:** IA653 – Natural Language Processing  
**Institution:** Clarkson University

---

## Project Overview

This notebook presents an end-to-end NLP pipeline applied to real-world job posting data scraped from ZipRecruiter. The project progressively builds from raw data ingestion through classical NLP, deep learning, and large language models.

### Topics Covered

| # | Module | Techniques |
|---|--------|-----------|
| 1 | Data Collection & EDA | Regex, JSON parsing, date extraction |
| 2 | Text Classification Baseline | CountVectorizer, Logistic Regression, cross-validation |
| 3 | Tokenization & Vocabulary | NLTK, BPE tokenizer, Herdan's Law |
| 4 | Word Vectors | TF-IDF, GloVe embeddings, Word2Vec, cosine similarity |
| 5 | Neural Networks | PyTorch feedforward networks, TF-IDF + dense layers |
| 6 | Language Generation | DistilGPT2 fine-tuning, beam search, top-p/top-k sampling |
| 7 | Named Entity Recognition | DistilBERT fine-tuning for skill tagging (BIO scheme) |
| 8 | Prompt Engineering & LLMs | AWS Bedrock + Mistral-7B, structured JSON extraction |
| 9 | RAG System | FAISS vector store, LangChain, Mistral-7B on Bedrock |

---


---
## Module 1: Data Collection & Exploratory Data Analysis

Parsing job postings from ZipRecruiter using regex and JSON. Tasks include extracting job metadata, identifying duplicate job IDs, acronym frequency analysis, and date extraction.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import os
import re
import json
from collections import Counter
import matplotlib.pyplot as plt
from datetime import datetime

In [ ]:
# Parse all search result folders and extract job URLs with metadata
base_dir = "data/search_results"
location, job_title, number, job_url = [], [], [], []

for textfolder in sorted(os.listdir(base_dir)):
    folder_path = os.path.join(base_dir, textfolder)
    if not os.path.isdir(folder_path):
        continue
    for file in sorted(os.listdir(folder_path)):
        if not file.endswith(".json"):
            continue
        text = Path(os.path.join(folder_path, file)).read_text(encoding="utf-8")
        urls = re.findall(r'"rawCanonicalZipJobPageUrl"\s*:\s*"([^"]+)"', text)
        l = len(urls)
        form_location = re.search(r'"form_location"\s*:\s*"([^"]+)"', text)
        form_location = form_location.group(1).lstrip(" ,") if form_location else None
        form_search = re.search(r'"form_search"\s*:\s*"([^"]+)"', text)
        form_search = form_search.group(1) if form_search else None
        page_match = re.search(r'"page_number"\s*:\s*([\d+]+)', text)
        page_match = page_match.group(1) if page_match else None
        job_url.extend(urls); location.extend([form_location]*l)
        job_title.extend([form_search]*l); number.extend([page_match]*l)

In [ ]:
# Extract job ID from URL using regex
def extract_jid(url):
    match = re.search(r"jid=(\w+)", url)
    return match.group(1) if match else None

df = pd.DataFrame({'job_link': job_url, 'search': job_title, 'location': location, 'page_number': number})
df['jid'] = df['job_link'].apply(extract_jid)
print(df.head())

In [ ]:
# Check for duplicate job IDs
dup = df.groupby('jid')['job_link'].count().reset_index(name='count').sort_values(by='count', ascending=False)
print(dup.head())
# jid is not unique — same job can appear in multiple search queries

In [ ]:
# Group by jid and aggregate search terms into a list
jid_table = df.groupby('jid')['search'].agg(list).reset_index()
print(jid_table.head())

In [ ]:
# Load assigned job files and extract description, company info, salary
with open('job_assignments_sampled.json', 'r') as f:
    file = json.load(f)
job_files = file['josea@clarkson.edu']['jobs']

job_descriptions, company_descriptions, salaries = [], [], []
for job in job_files:
    with open(f"data/jobs/{job}", 'r', encoding='utf-8') as f:
        job_json = json.load(f)
    job_descriptions.append(job_json.get('jobDetails', {}).get('Description'))
    salaries.append(job_json.get('jobDetails', {}).get('Salary'))
    company_descriptions.append(job_json.get('jobDetails', {}).get('CompanyDetails', {}).get('Description'))

job_df = pd.DataFrame({'jid': job_files, 'description': job_descriptions,
                        'company_description': company_descriptions, 'salary_range': salaries})
job_df['jid'] = job_df['jid'].str.replace('.json', '')
job_jid = pd.merge(job_df, jid_table, on='jid', how='left')
print(job_jid.head())

In [ ]:
# Acronym extraction — identify top technical skills in postings
job_jid['acronyms'] = job_jid['description'].apply(lambda x: re.findall(r'\b[A-Z]{3,4}\b', str(x)))
acronyms_all = [a for lst in job_jid['acronyms'] for a in lst]
top_10 = pd.Series(acronyms_all).value_counts().head(10)
print(top_10)
print('\nMost common acronyms: SQL, AWS, ETL, NLP, GCP, LLM, PTO, EEO, API, RAG')

In [ ]:
# Date extraction using multiple regex patterns
regex1 = r'\b([A-Z]\w+)\s?(\d{1,2})?,?\s(\d{4})\b'   # January 31, 2024
regex2 = r'\b([A-Z]\w*?.)\s?(\d{1,2})?,?\s(\d{4})\b'  # Jan 31, 2024
regex3 = r'(\d{2})((\/)|(\.))( \d{2})((\/)|(\.))( \d{4})'  # 10/10/2025
regex4 = r'(\d{4})\-(\d{2})\-(\d{2})'                    # 2025-08-09

job_jid['date_matches_01'] = job_jid['description'].apply(lambda x: re.findall(regex1, str(x)))
job_jid['date_matches_02'] = job_jid['description'].apply(lambda x: re.findall(regex2, str(x)))
job_jid['date_matches_03'] = job_jid['description'].apply(lambda x: re.findall(regex3, str(x)))
print(job_jid[['jid', 'date_matches_01', 'date_matches_02']].head())

In [ ]:
# Top 5 URLs extracted from job descriptions
url_regex = r'\bhttps://(?:www\.)?\w+\.(?:com|gov|org)\b'
urls = [u for desc in job_jid['description'] for u in re.findall(url_regex, str(desc))]
top_5_urls = [url for url, _ in Counter(urls).most_common(5)]
print('Top 5 URLs:', top_5_urls)

---
## Module 2: Text Classification Baseline

Using Bag-of-Words (CountVectorizer) and Logistic Regression to classify job postings by category. Evaluates 16 preprocessing scenarios combining lemmatization, stop word removal, n-grams, and binarization.


In [ ]:
import pandas as pd
import numpy as np
import time, dill, nltk, json, re, os
from pathlib import Path
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import product
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# Filter to jobs with exactly one label
filtered = job_jid[job_jid['search'].notna() & (job_jid['search'].apply(lambda x: isinstance(x, list) and len(x)==1))].copy()
filtered['description'] = filtered['description'].apply(lambda x: BeautifulSoup(x, "html.parser").get_text(" \n"))
filtered['search'] = filtered['search'].apply(lambda x: x[0])

# Class distribution
labels = filtered['search']
sns.countplot(x=labels, order=labels.value_counts().index)
plt.title('Job Category Distribution')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()
print('Labels are NOT evenly distributed — stratified splits required.')

In [ ]:
# Train/test split — stratified on label
X_train, X_test, y_train, y_test = train_test_split(
    filtered, filtered['search'], test_size=0.20, random_state=37, stratify=filtered['search'])

# Baseline: CountVectorizer + Logistic Regression
vectorizer = CountVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train['description'])
X_test_vec  = vectorizer.transform(X_test['description'])

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vec, y_train)
y_pred = lr.predict(X_test_vec)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred, average='macro'):.4f}")

In [ ]:
# Confusion matrix
metrics.ConfusionMatrixDisplay.from_predictions(y_test, y_pred, normalize='true', values_format='.0%')
plt.xticks(rotation=90)
plt.title('Baseline Confusion Matrix (Normalized)')
plt.tight_layout()
plt.show()

In [ ]:
# Text cleaning function for 16-scenario evaluation
def clean_texts(texts, lemmatize=False, remove_stopwords=False):
    lemmatizer = WordNetLemmatizer()
    eng_stopwords = set(stopwords.words('english'))
    token_pattern = r'(\b[\w]{2,}\b)'
    clean_docs = []
    for doc in texts:
        doc = doc.lower()
        tokens = nltk.regexp_tokenize(doc, token_pattern)
        if lemmatize:
            tokens = [lemmatizer.lemmatize(t) for t in tokens]
        if remove_stopwords:
            tokens = [t for t in tokens if t not in eng_stopwords]
        clean_docs.append(' '.join(tokens))
    return clean_docs

# Pre-compute 4 text variants to avoid redundant processing
X_train_list = X_train['description'].astype(str).tolist()
X_test_list  = X_test['description'].astype(str).tolist()

text_variants = {}
for lem in [True, False]:
    for sw in [True, False]:
        key = (lem, sw)
        text_variants[key] = (clean_texts(X_train_list, lem, sw),
                               clean_texts(X_test_list,  lem, sw))

In [ ]:
# Evaluate all 16 preprocessing scenarios
def evaluate_scenario(X_train_clean, X_test_clean, y_train, y_test, ngram, binary):
    pipe = Pipeline([
        ('vec', CountVectorizer(max_features=5000, ngram_range=(1, ngram), binary=binary)),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    cv_results = cross_validate(pipe, X_train_clean, y_train, cv=10, scoring='f1_macro')
    pipe.fit(X_train_clean, y_train)
    test_f1 = f1_score(y_test, pipe.predict(X_test_clean), average='macro')
    return {
        'mean_f1_cv': cv_results['test_score'].mean(),
        'median_f1_cv': np.median(cv_results['test_score']),
        'std_f1_cv': cv_results['test_score'].std(),
        'total_fit_time': cv_results['fit_time'].sum(),
        'test_f1': test_f1
    }

scenario_results = []
for (lem, sw), ngram, binary in product(text_variants.keys(), [1, 2], [True, False]):
    Xtr, Xte = text_variants[(lem, sw)]
    result = evaluate_scenario(Xtr, Xte, y_train, y_test, ngram, binary)
    result.update({'lemmatize': lem, 'remove_sw': sw, 'ngram': ngram, 'binary': binary})
    scenario_results.append(result)

results_df = pd.DataFrame(scenario_results).sort_values('test_f1', ascending=False)
print(results_df.head(5).to_string(index=False))

---
## Module 3: Tokenization & Vocabulary Analysis

Explores word tokenization, stemming vs. lemmatization, Herdan's Law (vocabulary growth), and training a Byte Pair Encoding (BPE) tokenizer from scratch on job descriptions.


In [ ]:
import json
from bs4 import BeautifulSoup
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
import re
from nltk.corpus import stopwords
from collections import Counter
from nltk.stem import PorterStemmer, WordNetLemmatizer
import seaborn as sns
import matplotlib.pyplot as plt
from tokenizers import (decoders, models, pre_tokenizers, processors, trainers, Tokenizer)

In [ ]:
# Load and extract text from first job description
with open('job_jid.json', 'r') as f:
    job_jid_df = pd.DataFrame(json.load(f))
job_jid_df['description'] = job_jid_df['description'].apply(
    lambda x: BeautifulSoup(x, "html.parser").get_text(" \n"))

first_desc = job_jid_df['description'].iloc[0]
tokenized = word_tokenize(first_desc)
print(f"Total tokens   : {len(tokenized)}")
print(f"Unique tokens  : {len(set(tokenized))}")
print(f"First 10 tokens: {tokenized[:10]}")

In [ ]:
# Clean tokens → remove punctuation → remove stopwords
cleaned = [t for t in tokenized if re.match(r'^[A-Za-z]+', t)]
lower   = [t.lower() for t in cleaned]
sw      = set(stopwords.words('english'))
filtered = [t for t in lower if t not in sw]
print('Top 15 words by frequency:')
print(Counter(filtered).most_common(15))

In [ ]:
# Stemming vs Lemmatization comparison
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stemmed    = [stemmer.stem(t) for t in filtered[:20]]
lemmatized = [lemmatizer.lemmatize(t) for t in filtered[:20]]
comparison = pd.DataFrame({'original': filtered[:20], 'stemmed': stemmed, 'lemmatized': lemmatized})
print(comparison)
print('\nStemming strips affixes aggressively (e.g. "running" → "run").')
print('Lemmatization returns the dictionary root form (context-aware).')

In [ ]:
# Herdan's Law — vocabulary growth as a function of corpus size
doc = job_jid_df['description'].values
texts = ' '.join(doc)
words = re.findall(r'\b[\w]+\b', texts.lower())

vocab, num_word, vocab_len = set(), [], []
for i, word in enumerate(words):
    vocab.add(word)
    if i % 100 == 0:
        num_word.append(i)
        vocab_len.append(len(vocab))

sns.scatterplot(x=num_word, y=vocab_len, color='steelblue', s=10)
plt.xlabel('Word Count'); plt.ylabel('Vocabulary Size')
plt.title("Herdan's Law — Vocabulary Growth (Regex Tokenizer)")
plt.grid(True); plt.show()
print('Vocabulary grows fast early on, then slows — new words become rarer as corpus grows.')

In [ ]:
# Train BPE tokenizer on job descriptions (vocab size = 30,000)
tokenizer_bpe = Tokenizer(models.BPE())
tokenizer_bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer = trainers.BpeTrainer(vocab_size=30000, special_tokens=["<|endoftext|>"])

def get_corpus():
    for desc in job_jid_df['description']:
        yield str(desc)

tokenizer_bpe.train_from_iterator(get_corpus(), trainer=trainer)
tokenizer_bpe.post_processor = processors.ByteLevel(trim_offsets=False)
print(f"BPE tokenizer trained. Vocab size: {tokenizer_bpe.get_vocab_size()}")

In [ ]:
# Compare Regex vs BPE vocabulary growth
bpe_enc = tokenizer_bpe.encode(texts)
bpe_vocab, bpe_words, bpe_lens = set(), [], []
for i, tok in enumerate(bpe_enc.tokens):
    bpe_vocab.add(tok)
    if i % 100 == 0:
        bpe_words.append(i); bpe_lens.append(len(bpe_vocab))

plt.plot(num_word, vocab_len, label='Regex tokenizer')
plt.plot(bpe_words, bpe_lens, label='BPE tokenizer')
plt.xlabel('Token Count'); plt.ylabel('Vocabulary Size')
plt.title('Vocabulary Growth: Regex vs BPE')
plt.legend(); plt.grid(True); plt.show()
print('BPE grows faster and plateaus near its maximum vocab size (~27k).')
print('Regex vocabulary grows without bound. BPE is more controlled.')

---
## Module 4: Word Vectors — TF-IDF, GloVe & Word2Vec

Represents job descriptions as TF-IDF vectors and dense word embeddings (GloVe, Word2Vec). Uses cosine similarity for semantic job search and nearest-neighbor retrieval.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader
from gensim.models import Word2Vec

In [ ]:
# Text cleaning pipeline
def clean_texts(texts, lemmatize=True, remove_stopwords=True, remove_punctuation=True):
    lemmatizer   = WordNetLemmatizer()
    eng_stopwords = set(stopwords.words('english'))
    token_pattern = r'(\b[\w]{2,}\b)'
    clean_docs = []
    for doc in texts:
        doc    = BeautifulSoup(doc, 'html.parser').get_text(' ')
        doc    = doc.lower()
        tokens = nltk.regexp_tokenize(doc, token_pattern)
        if remove_punctuation:
            tokens = [t for t in tokens if re.match(r'^[a-z]+', t)]
        if lemmatize:
            tokens = [lemmatizer.lemmatize(t) for t in tokens]
        if remove_stopwords:
            tokens = [t for t in tokens if t not in eng_stopwords]
        clean_docs.append(tokens)
    return clean_docs

df = pd.read_json('job_jid.json')
cleaned_des = clean_texts(df['description'])
print(f'Cleaned {len(cleaned_des)} job descriptions.')
print('Sample tokens:', cleaned_des[0][:15])

In [ ]:
# TF-IDF representation
cleaned_docs = [' '.join(d) for d in cleaned_des]
vectorizer   = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cleaned_docs)
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}  (jobs × terms)')

# Top 10 TF-IDF terms for first job
row           = tfidf_matrix[0].toarray().flatten()
feature_names = vectorizer.get_feature_names_out()
top_idx       = row.argsort()[::-1][:10]
print('\nTop 10 TF-IDF terms for job #1:')
for i in top_idx:
    print(f"  {feature_names[i]:<25} {row[i]:.4f}")

In [ ]:
# Pairwise cosine similarity (upper triangle only — remove self-comparisons)
cosine_sim  = cosine_similarity(tfidf_matrix)
i_u, j_u   = np.triu_indices_from(cosine_sim, k=1)
unique_sims = cosine_sim[i_u, j_u]
jid_list    = df['jid'].to_list()

df_sim = pd.DataFrame({'Job 1': [jid_list[i] for i in i_u],
                        'Job 2': [jid_list[j] for j in j_u],
                        'Similarity': unique_sims})
print('Top 10 most similar job pairs:')
print(df_sim.sort_values('Similarity', ascending=False).head(10).to_string(index=False))
print('\nNote: Similarity = 1.0 indicates duplicate job descriptions under different JIDs.')

In [ ]:
# GloVe semantic search
wiki_vectors = gensim.downloader.load('glove-wiki-gigaword-50')
print(f'GloVe embedding size: {wiki_vectors.vector_size}')
print(f"data ↔ information similarity : {wiki_vectors.similarity('data','information'):.4f}")
print(f"data ↔ banana similarity       : {wiki_vectors.similarity('data','banana'):.4f}")

def avg_glove(tokens, model):
    vecs = [model[w] for w in tokens if w in model]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

job_embeddings = [avg_glove(d, wiki_vectors) for d in cleaned_des]

# Semantic search query
query = 'machine learning engineering healthcare big data cloud'
query_vec  = avg_glove(query.split(), wiki_vectors).reshape(1, -1)
sims       = cosine_similarity(np.vstack(job_embeddings), query_vec).flatten()
top5_idx   = sims.argsort()[::-1][:5]
print('\nTop 5 jobs most relevant to query:')
for i in top5_idx:
    print(f"  [{i}] {df.loc[i,'search']}")

In [ ]:
# Train Word2Vec models (CBOW and Skip-gram)
cbow_model     = Word2Vec(sentences=cleaned_des, vector_size=100, window=5, min_count=2, sg=0)
skipgram_model = Word2Vec(sentences=cleaned_des, vector_size=100, window=5, min_count=2, sg=1)
print(f'CBOW vocab size    : {len(cbow_model.wv)}')
print(f'Skip-gram vocab size: {len(skipgram_model.wv)}')

---
## Module 5: Neural Networks with PyTorch

Builds a configurable feedforward neural network on top of TF-IDF features to classify job postings. Implements a full training loop with validation, learning curves, and a confusion matrix.


In [ ]:
import json, re
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Load data and filter to single-label jobs
all_jobs = pd.read_json('job_jid.json')
df = all_jobs[all_jobs['search'].notna()]
df = df[df['search'].apply(lambda x: isinstance(x, list) and len(x) == 1)].copy()
df['label'] = df['search'].apply(lambda x: x[0])
texts  = df['description'].to_list()
labels = df['label'].to_list()
encoded_labels = LabelEncoder().fit_transform(labels)
print(f'Dataset: {len(df)} samples, {len(set(labels))} classes')

In [ ]:
# Text cleaning
def clean_texts(texts, remove_stopwords=True):
    eng_sw = set(stopwords.words('english'))
    cleaned = []
    for doc in texts:
        doc = BeautifulSoup(doc, 'html.parser').get_text(' ').lower()
        doc = re.sub(r'[^a-z\s]', '', doc)
        if remove_stopwords:
            doc = ' '.join(w for w in doc.split() if w not in eng_sw)
        cleaned.append(doc)
    return cleaned

X_cleaned = clean_texts(texts)

In [ ]:
# Stratified train / validation / test split
X_tr_full, X_test, y_tr_full, y_test = train_test_split(
    X_cleaned, encoded_labels, train_size=0.80, stratify=encoded_labels, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_tr_full, y_tr_full, train_size=0.80, stratify=y_tr_full, random_state=42)
print(f'Train: {len(X_train)}  |  Valid: {len(X_valid)}  |  Test: {len(X_test)}')

In [ ]:
# TF-IDF → PyTorch DataLoaders
def prep_tfidf(X_train, X_valid, X_test, y_train, y_valid, y_test,
               batch_size=32, max_features=10000, ngram_range=(1,2)):
    vec = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    Xtr = torch.tensor(vec.fit_transform(X_train).toarray(), dtype=torch.float32)
    Xva = torch.tensor(vec.transform(X_valid).toarray(),  dtype=torch.float32)
    Xte = torch.tensor(vec.transform(X_test).toarray(),   dtype=torch.float32)
    def loader(X, y): return DataLoader(
        TensorDataset(X, torch.tensor(y, dtype=torch.long)), batch_size=batch_size)
    return loader(Xtr, y_train), loader(Xva, y_valid), loader(Xte, y_test)

train_loader, valid_loader, test_loader = prep_tfidf(
    X_train, X_valid, X_test, y_train, y_valid, y_test)

In [ ]:
# Feedforward neural network with configurable depth and dropout
class DenseTFIDFModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_hidden=1, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        if dropout > 0: layers.append(nn.Dropout(dropout))
        for _ in range(n_hidden):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)

model = DenseTFIDFModel(input_dim=10000, hidden_dim=256,
                         output_dim=len(set(labels)), n_hidden=1, dropout=0.3)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
# Training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[]}

for epoch in range(20):
    model.train()
    all_preds, all_labels_ep, ep_loss = [], [], 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out  = model(Xb)
        loss = criterion(out, yb)
        loss.backward(); optimizer.step()
        ep_loss += loss.item()
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels_ep.extend(yb.cpu().numpy())

    model.eval()
    val_preds, val_true, val_loss = [], [], 0.0
    with torch.no_grad():
        for Xb, yb in valid_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out = model(Xb)
            val_loss += criterion(out, yb).item()
            val_preds.extend(out.argmax(1).cpu().numpy())
            val_true.extend(yb.cpu().numpy())

    history['train_loss'].append(ep_loss / len(train_loader))
    history['val_loss'].append(val_loss / len(valid_loader))
    history['train_f1'].append(metrics.f1_score(all_labels_ep, all_preds, average='macro'))
    history['val_f1'].append(metrics.f1_score(val_true, val_preds, average='macro'))
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Train F1: {history['train_f1'][-1]:.4f} | Val F1: {history['val_f1'][-1]:.4f}")

In [ ]:
# Learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Validation')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_f1'], label='Train'); ax2.plot(history['val_f1'], label='Validation')
ax2.set_title('F1 Score (Macro)'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.suptitle('Training History — Feedforward Neural Network'); plt.tight_layout(); plt.show()

---
## Module 6: Language Generation — Fine-tuning DistilGPT2

Fine-tunes DistilGPT2 on job descriptions for causal language modeling. Compares three generation strategies (beam search, top-p, top-k) using sequence log-probabilities.

> ⚠️ **Note:** This module was run on Google Colab with GPU acceleration.


In [ ]:
import pandas as pd
import json
from bs4 import BeautifulSoup
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForCausalLM, TrainingArguments, Trainer
import torch
import torch.nn.functional as F

In [ ]:
# Load and sample 1,000 job descriptions
all_jobs = pd.read_json('job_jid.json').sample(n=1000, random_state=42)
all_jobs['cleaned_description'] = all_jobs['description'].apply(
    lambda x: BeautifulSoup(x, 'html.parser').get_text(' '))

# Build HuggingFace Dataset with 80/10/10 split
dataset    = Dataset.from_pandas(all_jobs[['cleaned_description']])
split      = dataset.train_test_split(test_size=0.2)
val_test   = split['test'].train_test_split(test_size=0.5)
dataset    = DatasetDict({'train': split['train'], 'val': val_test['train'], 'test': val_test['test']})
print(dataset)

In [ ]:
# Tokenise with DistilGPT2 and chunk into blocks of 128 tokens
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilgpt2')
tokenizer.pad_token = tokenizer.eos_token

def preprocess(examples):
    return tokenizer([x + tokenizer.eos_token for x in examples['cleaned_description']])

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset['train'].column_names)

BLOCK_SIZE = 128
def group_texts(examples):
    cat = {k: sum(examples[k], []) for k in examples}
    total = (len(cat[list(cat.keys())[0]]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {k: [t[i:i+BLOCK_SIZE] for i in range(0, total, BLOCK_SIZE)] for k, t in cat.items()}
    result['labels'] = result['input_ids'].copy()
    return result

lm_dataset   = tokenized.map(group_texts, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
# Load pre-trained DistilGPT2 and generate text with 3 strategies
model = AutoModelForCausalLM.from_pretrained('distilbert/distilgpt2')

prompt   = "Finish this job posting: Come join Acme Industries and our team of Data Scientists"
inputs   = tokenizer(prompt, return_tensors='pt')
max_new  = model.config.n_positions - inputs['input_ids'].shape[1]

# Beam search
out_beam = model.generate(**inputs, max_new_tokens=max_new, num_beams=5,
                            no_repeat_ngram_size=1, do_sample=True,
                            eos_token_id=tokenizer.eos_token_id)
# Top-p sampling
out_topp = model.generate(**inputs, max_new_tokens=max_new, do_sample=True,
                            top_p=0.85, eos_token_id=tokenizer.eos_token_id)
# Top-k sampling
out_topk = model.generate(**inputs, max_new_tokens=max_new, do_sample=True,
                            top_k=25,  eos_token_id=tokenizer.eos_token_id)

for name, out in [('Beam search', out_beam), ('Top-p (0.85)', out_topp), ('Top-k (25)', out_topk)]:
    print(f'--- {name} ---')
    print(tokenizer.decode(out[0], skip_special_tokens=True)[:300])
    print()

In [ ]:
# Fine-tune for 3 epochs
training_args = TrainingArguments(
    output_dir='distilgpt2-jobs', report_to='none', eval_strategy='epoch',
    num_train_epochs=3, per_device_train_batch_size=4, per_device_eval_batch_size=8,
    gradient_accumulation_steps=1, fp16=True, learning_rate=2e-5,
    weight_decay=0.01, save_strategy='epoch')

trainer = Trainer(model=model, args=training_args,
                   train_dataset=lm_dataset['train'], eval_dataset=lm_dataset['val'],
                   data_collator=data_collator, processing_class=tokenizer)
trainer.train()
model.save_pretrained('./distilgpt2-jobs')
tokenizer.save_pretrained('./distilgpt2-jobs')

---
## Module 7: Named Entity Recognition — Skill Tagging with DistilBERT

Fine-tunes DistilBERT for token classification using BIO tagging to identify skills in job descriptions. Labels: `O`, `B-SKILL`, `I-SKILL`.

> ⚠️ **Note:** This module was run on Google Colab with GPU acceleration.


In [ ]:
from datasets import load_from_disk, DatasetDict
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                           TrainingArguments, Trainer, DataCollatorForTokenClassification, pipeline)
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from evaluate import load
import pandas as pd

In [ ]:
# Load NER dataset and split 80/10/10
data      = load_from_disk('ner_dataset').train_test_split(test_size=0.2, seed=42)
temp      = data['test'].train_test_split(test_size=0.5, seed=42)
data      = DatasetDict({'train': data['train'], 'validation': temp['train'], 'test': temp['test']})
label_list = ['O', 'B-SKILL', 'I-SKILL']
print(data)
print('Labels:', label_list)

In [ ]:
# Tokenize and align BIO labels with subword tokens
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)
    labels = []
    for idx, label in enumerate(examples['ner_tags']):
        word_ids, prev, label_ids = tokenized_inputs.word_ids(batch_index=idx), None, []
        for word_idx in word_ids:
            if word_idx is None or word_idx == prev:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_idx])
            prev = word_idx
        labels.append(label_ids)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

tokenized_data = data.map(tokenize_and_align_labels, batched=True)

In [ ]:
# seqeval metric for NER evaluation
seqeval    = load('seqeval')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds  = [[label_list[p] for p, l in zip(pred, lab) if l != -100]
                    for pred, lab in zip(predictions, labels)]
    true_labels = [[label_list[l] for p, l in zip(pred, lab) if l != -100]
                    for pred, lab in zip(predictions, labels)]
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {'precision': results['overall_precision'], 'recall': results['overall_recall'],
            'f1': results['overall_f1'], 'accuracy': results['overall_accuracy']}

In [ ]:
# Load DistilBERT for token classification and train for 5 epochs
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    'distilbert/distilbert-base-uncased',
    num_labels=len(label_list), id2label=id2label, label2id=label2id)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='ner-distilbert', eval_strategy='epoch', num_train_epochs=5,
    save_strategy='epoch', logging_strategy='epoch', learning_rate=1e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    weight_decay=0.01, report_to='none')

trainer = Trainer(model=model, args=training_args,
                   train_dataset=tokenized_data['train'],
                   eval_dataset=tokenized_data['validation'],
                   data_collator=data_collator, tokenizer=tokenizer,
                   compute_metrics=compute_metrics)
trainer.train()
trainer.save_model('./ner-distilbert')
tokenizer.save_pretrained('./ner-distilbert')

In [ ]:
# Run skill extraction on a sample job description
ner_pipeline = pipeline('token-classification',
                          model='./ner-distilbert',
                          tokenizer='./ner-distilbert',
                          aggregation_strategy='simple')

sample_text = """Required Skills: Python, SQL, and experience with machine learning frameworks
                 such as TensorFlow and PyTorch. Strong communication skills required."""
preds = ner_pipeline(sample_text)
print('Extracted skills:')
for p in preds:
    if p['entity_group'] != 'O':
        print(f"  {p['word']:<30} [{p['entity_group']}]  score={p['score']:.3f}")

---
## Module 8: Prompt Engineering & LLM Skill Extraction

Uses prompt engineering with Mistral-7B (via AWS Bedrock) to extract structured skill information from job descriptions. Returns JSON with `programming_languages`, `technical_skills`, and `other` fields.


In [ ]:
import json, re, boto3, pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from transformers import AutoTokenizer

In [ ]:
# Load 10 job descriptions for this module
jobs = pd.read_json('homework_08_jobs.json')
clean_desc = jobs['job_description'].apply(
    lambda x: re.sub(r'\s+', ' ', BeautifulSoup(str(x), 'html.parser').get_text()).strip())
print(f'Loaded {len(clean_desc)} job descriptions')

In [ ]:
# Prompt template — instructs model to return only valid JSON
job_prompt = """<s> Given job description {job}

[INST] Please extract the job skills and provide in the JSON format with the following structure
{sample_output} DO NOT INCLUDE ANY OTHER TEXT OTHER THAN THE JSON RESPONSE.
If there are no skills just return an empty JSON object.[/INST]

"""

json_structure = {
    'programming_languages': ['Python', 'R', 'SQL'],
    'technical skills': ['machine learning', 'data science'],
    'other': ['communication skills']
}

In [ ]:
# Tokenize and estimate API cost per prompt
tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')
costs = {'input': 0.00015, 'output': 0.00020}

def get_cost(input_tokens, output_tokens, costs):
    return (input_tokens/1000)*costs['input'] + (output_tokens/1000)*costs['output']

total_cost = 0
for job in clean_desc:
    prompt  = job_prompt.format(job=job, sample_output=json_structure)
    in_tok  = len(tokenizer(prompt)['input_ids'])
    out_tok = len(tokenizer(str(json_structure))['input_ids'])
    total_cost += get_cost(in_tok, out_tok, costs)

print(f'Estimated total cost for 10 prompts: ${total_cost:.6f}')

In [ ]:
# Invoke Mistral-7B on AWS Bedrock and parse JSON responses
client   = boto3.client(service_name='bedrock-runtime', region_name='us-east-1')
model_id = 'mistral.mistral-7b-instruct-v0:2'
responses, parsed_resp = [], []

for job in clean_desc:
    prompt   = job_prompt.format(job=job, sample_output=json_structure)
    body     = {'prompt': prompt, 'max_tokens': 512, 'temperature': 0.5}
    response = client.invoke_model(modelId=model_id, body=json.dumps(body))
    responses.append(response['body'].read().decode('utf-8'))

# Parse JSON from each response
pattern = re.compile(r'\{.*\}', re.DOTALL | re.MULTILINE)
for idx, resp in enumerate(responses):
    match = pattern.search(resp)
    if not match:
        print(f'Job {idx}: No JSON found'); parsed_resp.append({}); continue
    try:
        parsed_resp.append(json.loads(match.group(0).strip()))
    except Exception:
        print(f'Job {idx}: JSON parse error'); parsed_resp.append({})

print(f'Successfully parsed {sum(1 for r in parsed_resp if r)} / {len(responses)} responses')

In [ ]:
# Hallucination check — verify extracted languages appear in the job text
print('Checking for hallucinated programming languages...')
for idx, parsed in enumerate(parsed_resp):
    langs    = parsed.get('programming_languages', [])
    job_text = clean_desc[idx].lower()
    if langs:
        missing = [l for l in langs if l.lower() not in job_text]
        if missing:
            print(f'  Job {idx}: {missing} not found in description')
print('Check complete.')

---
## Module 9: Retrieval-Augmented Generation (RAG)

Builds a complete RAG pipeline: chunking job descriptions with LangChain, encoding with sentence-transformers, indexing in FAISS, and querying with Mistral-7B on AWS Bedrock.


In [ ]:
import pandas as pd, json, re, os, boto3
from bs4 import BeautifulSoup
from transformers import AutoTokenizer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_aws import ChatBedrock
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# Load and sample 500 jobs
all_jobs = pd.read_json('job_jid.json').sample(n=500, random_state=42)

def clean_text(text):
    return re.sub(r'\s+', ' ', BeautifulSoup(str(text), 'html.parser').get_text(' ')).strip()

all_jobs['cleaned_description']         = all_jobs['description'].apply(clean_text)
all_jobs['cleaned_company_description'] = all_jobs['company_description'].apply(clean_text)
all_jobs['combined_text'] = (all_jobs['search'].astype(str) + ' ' +
                              all_jobs['salary_range'].astype(str) + ' ' +
                              all_jobs['cleaned_description'] + ' ' +
                              all_jobs['cleaned_company_description'])
print(f'Dataset shape: {all_jobs.shape}')

In [ ]:
# Token statistics and chunk size estimation
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
all_jobs['token_count'] = all_jobs['combined_text'].apply(
    lambda x: len(tokenizer.encode(str(x), truncation=False)))
all_jobs['char_count'] = all_jobs['combined_text'].apply(len)

print(f"Mean tokens: {all_jobs['token_count'].mean():.0f}")
print(f"Max tokens : {all_jobs['token_count'].max()}")

import matplotlib.pyplot as plt
plt.scatter(all_jobs['char_count'], all_jobs['token_count'], alpha=0.4, edgecolors='k', s=15)
plt.xlabel('Characters'); plt.ylabel('Tokens')
plt.title('Characters vs Tokens per Job Description')
plt.tight_layout(); plt.show()
print('Chunk size of 2000 characters avoids tokenizer truncation for most postings.')

In [ ]:
# Chunk documents with metadata
splitter  = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)
documents = []
for _, row in all_jobs.iterrows():
    for chunk in splitter.split_text(row['combined_text']):
        documents.append(Document(
            page_content=chunk,
            metadata={'jid': row.get('jid'),
                       'job_title': ' '.join(row['search']) if isinstance(row['search'], list) else row.get('search'),
                       'job_salary': row.get('salary_range')}))
print(f'Total document chunks: {len(documents)}')

In [ ]:
# Build FAISS vector index with sentence-transformers embeddings
FAISS_DIR  = 'faiss_index'
os.makedirs(FAISS_DIR, exist_ok=True)
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
faiss_idx  = FAISS.from_documents(documents, embeddings)
faiss_idx.save_local(FAISS_DIR)
print('FAISS index built and saved.')

# Reload and create retriever (k=10)
vectordb  = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
retriever = vectordb.as_retriever(search_kwargs={'k': 10})

In [ ]:
# Build RAG chain: retriever → prompt → Mistral-7B → answer
bedrock_rt = boto3.Session(region_name='us-east-1').client('bedrock-runtime')
llm = ChatBedrock(client=bedrock_rt,
                   model_id='mistral.mistral-7b-instruct-v0:2',
                   model_kwargs={'temperature': 0.2, 'max_tokens': 700})

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise assistant. Answer ONLY using the provided context. '
               "If the answer is not in the context, say 'I don't know.'\n\n[CONTEXT]\n{context}"),
    ('human', '{input}')
])

def format_docs(docs): return '\n\n'.join(d.page_content for d in docs)

rag_chain = ({'input': RunnablePassthrough(), 'context': retriever | format_docs}
              | prompt | llm | StrOutputParser())

In [ ]:
# Query the RAG system
queries = [
    'Which postings mention remote work and how often may I work remotely?',
    'List companies hiring for NLP roles.',
    "What's the highest salary mentioned for Data Scientist positions?"
]

for i, query in enumerate(queries, 1):
    print(f'\nQuery {i}: {query}')
    answer  = rag_chain.invoke(query)
    print(f'Answer: {answer}')
    results = retriever.get_relevant_documents(query)
    print('Top retrieved jobs:')
    for doc in results[:3]:
        print(f"  - {doc.metadata.get('job_title','N/A')} | Salary: {doc.metadata.get('job_salary','N/A')}")

---
## Summary & Key Takeaways

| Module | Key Result |
|--------|-----------|
| Data Collection | Parsed 50k+ job postings via regex; extracted acronyms, dates, URLs |
| Text Classification | Logistic Regression baseline ~83% F1; best scenario with lemmatization + bigrams |
| Tokenization | Demonstrated Herdan's Law; BPE vocab plateaus near max size |
| Word Vectors | GloVe semantic search outperforms TF-IDF for conceptual queries |
| Neural Networks | PyTorch feedforward net improves over baseline with dense TF-IDF features |
| Language Generation | Fine-tuned DistilGPT2; top-k sampling produced best log-probability |
| NER / Skill Tagging | DistilBERT fine-tuned for BIO skill extraction; best on `O`, struggles on `I-SKILL` |
| Prompt Engineering | Mistral-7B extracted structured JSON skills with no hallucinated languages |
| RAG System | FAISS + Mistral-7B accurately retrieves relevant postings; salary range less reliable |

---

*Tools: Python · NLTK · HuggingFace Transformers · PyTorch · Gensim · LangChain · AWS Bedrock · FAISS*
